# Работа с данными

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)


In [2]:
path = 'data_raw/BTCUSDT_futures_um_1d_with_funding_2020-01_2025-12.csv'

df = pd.read_csv(path)
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], utc=True, errors='coerce')
df = df.sort_values('datetime_utc').reset_index(drop=True)
df.shape

(2192, 13)

In [3]:
df.head(3)

,datetime_utc,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote,date_utc,funding_rate_day_sum,funding_points_in_day
0,2020-01-01 00:00:00+00:00,7189.43,7260.43,7170.15,7197.57,56801.329,4.096788e+08,101871,28834.200,2.080020e+08,2020-01-01,-0.000344,3
1,2020-01-02 00:00:00+00:00,7197.57,7209.59,6922.00,6962.04,115295.677,8.156278e+08,224747,55404.262,3.919117e+08,2020-01-02,0.000138,3
2,2020-01-03 00:00:00+00:00,6962.34,7407.28,6863.44,7341.72,208493.458,1.507314e+09,409820,107485.965,7.770568e+08,2020-01-03,0.000068,3


In [4]:
df.tail(3)

,datetime_utc,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote,date_utc,funding_rate_day_sum,funding_points_in_day
2189,2025-12-29 00:00:00+00:00,87920.4,90373.4,86760.0,87203.3,190004.209,1.678775e+10,3903591,95283.174,8.416034e+09,2025-12-29,0.000248,3
2190,2025-12-30 00:00:00+00:00,87203.2,89355.0,86789.0,88455.3,126745.260,1.117228e+10,2751660,62853.527,5.540337e+09,2025-12-30,0.000230,3
2191,2025-12-31 00:00:00+00:00,88455.2,89192.8,87189.2,87608.2,96768.357,8.534611e+09,2111873,48482.035,4.275528e+09,2025-12-31,0.000081,3


In [5]:
checks = {}

checks['rows'] = len(df)
checks['datetime_nulls'] = int(df['datetime_utc'].isna().sum())
checks['datetime_duplicates'] = int(df['datetime_utc'].duplicated().sum())
checks['is_sorted'] = bool(df['datetime_utc'].is_monotonic_increasing)

dt = df['datetime_utc'].dropna()
if len(dt) > 1:
    gaps = dt.diff().dt.total_seconds().div(86400)
    gap_counts = gaps.value_counts(dropna=True).sort_index()
    checks['min_gap_days'] = float(gaps.min())
    checks['max_gap_days'] = float(gaps.max())
else:
    gap_counts = pd.Series(dtype='int64')

checks, gap_counts.head(10), gap_counts.tail(10)


({'rows': 2192,
  'datetime_nulls': 0,
  'datetime_duplicates': 0,
  'is_sorted': True,
  'min_gap_days': 1.0,
  'max_gap_days': 1.0},
 datetime_utc
 1.0    2191
 Name: count, dtype: int64,
 datetime_utc
 1.0    2191
 Name: count, dtype: int64)

In [6]:
num_cols = [
    'open', 'high', 'low', 'close', 'volume',
    'quote_volume', 'taker_buy_base', 'taker_buy_quote', 'funding_rate_day_sum'
]
int_cols = ['trades']

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['trades'] = pd.to_numeric(df['trades'], errors='coerce').astype('Int64')

before = len(df)
df = df.dropna(subset=['datetime_utc', 'open', 'high', 'low', 'close']).copy()
after = len(df)

bad_high = (df['high'] < df[['open', 'close']].max(axis=1)).sum()
bad_low = (df['low'] > df[['open', 'close']].min(axis=1)).sum()
nonpos_close = (df['close'] <= 0).sum()

{
    'dropped_rows_due_to_nans': before - after,
    'bad_high_count': int(bad_high),
    'bad_low_count': int(bad_low),
    'nonpositive_close_count': int(nonpos_close)
}


{'dropped_rows_due_to_nans': 0,
 'bad_high_count': 0,
 'bad_low_count': 0,
 'nonpositive_close_count': 0}

Доходность, лог-доходность, дневной ценовой размах, True Range, Average True Range, Годовая волатильность на основе доходностей

In [7]:
df = df.set_index('datetime_utc').sort_index()

df['ret_1d'] = df['close'].pct_change()
df['logret_1d'] = np.log(df['close']).diff()

df['range'] = (df['high'] - df['low']) / df['close'].shift(1)

prev_close = df['close'].shift(1)
tr = pd.concat(
    [
        (df['high'] - df['low']),
        (df['high'] - prev_close).abs(),
        (df['low'] - prev_close).abs()
    ],
    axis=1
).max(axis=1)

df['true_range'] = tr
df['atr_14'] = df['true_range'].rolling(14, min_periods=14).mean()

df['vol_30'] = df['ret_1d'].rolling(30, min_periods=30).std() * np.sqrt(365)

df[['open', 'high', 'low', 'close', 'volume', 'ret_1d', 'logret_1d', 'atr_14', 'vol_30']].tail(10)


,open,high,low,close,volume,ret_1d,logret_1d,atr_14,vol_30
datetime_utc,,,,,,,,,
2025-12-22 00:00:00+00:00,88621.3,90599.0,87845.0,88581.5,154693.469,-0.000449,-0.000449,3353.507143,0.409120
2025-12-23 00:00:00+00:00,88581.5,88904.9,86536.0,87463.0,133657.948,-0.012627,-0.012707,3152.278571,0.403170
2025-12-24 00:00:00+00:00,87463.1,88033.3,86355.0,87627.3,81543.508,0.001879,0.001877,3063.321429,0.399013
2025-12-25 00:00:00+00:00,87627.4,88594.0,86902.1,87182.8,57796.252,-0.005073,-0.005086,2873.650000,0.397680
2025-12-26 00:00:00+00:00,87182.8,89557.6,86619.9,87318.2,149971.737,0.001553,0.001552,2848.485714,0.376589
2025-12-27 00:00:00+00:00,87318.3,87956.9,87203.0,87829.4,27294.109,0.005854,0.005837,2838.971429,0.375544
2025-12-28 00:00:00+00:00,87829.5,88078.3,87402.6,87920.5,37359.964,0.001037,0.001037,2677.542857,0.375364
2025-12-29 00:00:00+00:00,87920.4,90373.4,86760.0,87203.3,190004.209,-0.008157,-0.008191,2585.685714,0.376215
2025-12-30 00:00:00+00:00,87203.2,89355.0,86789.0,88455.3,126745.260,0.014357,0.014255,2560.550000,0.379778


In [8]:
df[['open', 'high', 'low', 'close', 'volume', 'ret_1d', 'logret_1d', 'atr_14', 'vol_30']].head(31)

,open,high,low,close,volume,ret_1d,logret_1d,atr_14,vol_30
datetime_utc,,,,,,,,,
2020-01-01 00:00:00+00:00,7189.43,7260.43,7170.15,7197.57,56801.329,NaN,NaN,NaN,NaN
2020-01-02 00:00:00+00:00,7197.57,7209.59,6922.00,6962.04,115295.677,-0.032724,-0.033271,NaN,NaN
2020-01-03 00:00:00+00:00,6962.34,7407.28,6863.44,7341.72,208493.458,0.054536,0.053101,NaN,NaN
2020-01-04 00:00:00+00:00,7341.60,7400.00,7269.21,7350.71,92586.033,0.001225,0.001224,NaN,NaN
2020-01-05 00:00:00+00:00,7350.54,7495.00,7303.00,7354.36,117765.972,0.000497,0.000496,NaN,NaN
2020-01-06 00:00:00+00:00,7354.36,7808.65,7345.00,7757.39,168150.317,0.054802,0.053353,NaN,NaN
2020-01-07 00:00:00+00:00,7757.74,8215.33,7733.00,8152.49,280809.162,0.050932,0.049677,NaN,NaN
2020-01-08 00:00:00+00:00,8150.90,8468.42,7870.11,8059.84,321225.114,-0.011365,-0.011430,NaN,NaN
2020-01-09 00:00:00+00:00,8059.77,8060.00,7750.01,7818.59,187263.143,-0.029932,-0.030389,NaN,NaN


In [9]:
keep_cols = [
    'open', 'high', 'low', 'close', 'volume',
    'quote_volume', 'trades', 'taker_buy_base', 'taker_buy_quote',
    'ret_1d', 'logret_1d', 'range', 'true_range', 'atr_14', 'vol_30', 'funding_rate_day_sum'
]

df_bt = df[keep_cols].copy()

required = ['ret_1d', 'atr_14', 'vol_30']
df_bt_clean = df_bt.dropna(subset=required).copy()

df_bt_clean.shape

(2162, 16)

In [10]:
out_path = 'data/btc_futures_2020-2025.csv'
df_bt_clean.to_csv(out_path, index=True)
out_path

'data/btc_futures_2020-2025.csv'

In [12]:
df_bt_clean

,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote,ret_1d,logret_1d,range,true_range,atr_14,vol_30,funding_rate_day_sum
datetime_utc,,,,,,,,,,,,,,,,
2020-01-31 00:00:00+00:00,9525.67,9550.72,9220.00,9364.51,146644.026,1.373299e+09,266727,70753.403,6.625507e+08,-0.016918,-0.017063,0.034719,330.72,344.225714,0.593523,0.001706
2020-02-01 00:00:00+00:00,9364.50,9477.76,9306.00,9392.74,109040.285,1.024211e+09,229569,54324.089,5.103520e+08,0.003015,0.003010,0.018342,171.76,343.708571,0.574485,0.001897
2020-02-02 00:00:00+00:00,9392.99,9489.92,9160.01,9338.20,148402.977,1.391949e+09,285872,72902.026,6.840480e+08,-0.005807,-0.005824,0.035124,329.91,312.674286,0.554422,0.001287
2020-02-03 00:00:00+00:00,9340.49,9647.61,9250.00,9300.64,159049.385,1.491077e+09,313233,79180.283,7.425274e+08,-0.004022,-0.004030,0.042579,397.61,325.052857,0.555583,0.001065
2020-02-04 00:00:00+00:00,9300.63,9368.00,9105.00,9198.35,147843.902,1.363688e+09,282503,71776.668,6.622206e+08,-0.010998,-0.011059,0.028278,263.00,319.445714,0.559044,0.001919
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-27 00:00:00+00:00,87318.30,87956.90,87203.00,87829.40,27294.109,2.388675e+09,702865,13546.965,1.185725e+09,0.005854,0.005837,0.008634,753.90,2838.971429,0.375544,0.000104
2025-12-28 00:00:00+00:00,87829.50,88078.30,87402.60,87920.50,37359.964,3.278206e+09,886145,19229.166,1.687448e+09,0.001037,0.001037,0.007693,675.70,2677.542857,0.375364,0.000174
2025-12-29 00:00:00+00:00,87920.40,90373.40,86760.00,87203.30,190004.209,1.678775e+10,3903591,95283.174,8.416034e+09,-0.008157,-0.008191,0.041098,3613.40,2585.685714,0.376215,0.000248
